# Unsupervised Machine Learning — Final Project Notebook
**Course:** Unsupervised Machine Learning (IBM Skills Network / Coursera)  
**Audience:** Chief Data Officer (CDO) & Senior Analytics Stakeholders  
**Dataset:** UCI / Scikit-Learn Enological Wine Physicochemical Dataset ($N = 178$, $p = 13$ raw + $3$ engineered = $16$ features)

---
## 1. Main Objective of the Analysis
The objective of this study is to combine **Dimensionality Reduction (PCA, t-SNE)** with **Unsupervised Clustering (K-Means, Agglomerative Hierarchical Clustering, DBSCAN, and Gaussian Mixture Models)** to discover intrinsic physicochemical product segments without relying on manual sensory grading labels, isolate anomalous batches, and evaluate whether cluster-specific regression improves downstream prediction relative to a single global model.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from scipy.cluster.hierarchy import linkage
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score, r2_score, mean_squared_error
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict

np.random.seed(42)
raw = load_wine()
df = pd.DataFrame(raw.data, columns=raw.feature_names)
true_cultivars = raw.target  # Strictly withheld from training; used only for post-hoc external validation
print(f'Dataset shape: {df.shape}, Missing values: {df.isna().sum().sum()}')

## 2. Data Exploration, IQR Outlier Winsorization & Domain Feature Engineering
We inspect feature distributions, Winsorize 21 extreme univariate outliers at $1.5 \times \text{IQR}$ fences, engineer 3 biochemical ratios (`flavonoid_phenol_ratio`, `chromatic_density_index`, `mineral_alkalinity_ratio`), and standardize all 16 features via `StandardScaler`.

In [2]:
df_clean = df.copy()
for col in raw.feature_names:
    q25, q75 = np.percentile(df_clean[col], [25, 75])
    iqr = q75 - q25
    df_clean[col] = np.clip(df_clean[col], q25 - 1.5 * iqr, q75 + 1.5 * iqr)

df_clean['flavonoid_phenol_ratio'] = df_clean['flavanoids'] / (df_clean['total_phenols'] + 1e-6)
df_clean['chromatic_density_index'] = df_clean['color_intensity'] / (df_clean['hue'] + 1e-6)
df_clean['mineral_alkalinity_ratio'] = df_clean['alcalinity_of_ash'] / (df_clean['ash'] + 1e-6)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean.values)
print('Standardized feature matrix shape:', X_scaled.shape)

## 3. Dimensionality Reduction: Principal Component Analysis (PCA) & t-SNE
PC1 explains **37.02%** of variance (Polyphenolic Structure Axis), PC2 explains **20.76%** (Chromatic Body & Ripeness Axis), and the top 5 PCs capture **79.48%** of total variance.

In [3]:
pca = PCA(random_state=42)
X_pca = pca.fit_transform(X_scaled)
for i in range(5):
    print(f'PC{i+1}: Eigenvalue={pca.explained_variance_[i]:.4f}, '
          f'Explained={pca.explained_variance_ratio_[i]*100:.2f}%, '
          f'Cumulative={np.cumsum(pca.explained_variance_ratio_)[i]*100:.2f}%')

## 4. Comparing Four Unsupervised Clustering Model Families
We compare **K-Means**, **Agglomerative Hierarchical Clustering**, **DBSCAN**, and **Gaussian Mixture Models (GMM)** across hyperparameter grids.

In [4]:
models = {
    'K-Means (16D, k=3)': KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(X_scaled),
    'PCA-5D + K-Means (k=3)': KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(X_pca[:, :5]),
    'Hierarchical Complete (k=3)': AgglomerativeClustering(n_clusters=3, linkage='complete').fit_predict(X_scaled),
    'Hierarchical Ward (k=3)': AgglomerativeClustering(n_clusters=3, linkage='ward').fit_predict(X_scaled),
    'GMM Full Covariance (k=3)': GaussianMixture(n_components=3, covariance_type='full', n_init=10, random_state=42).fit_predict(X_scaled),
}

for name, labels in models.items():
    space = X_pca[:, :5] if 'PCA-5D' in name else X_scaled
    sil = silhouette_score(space, labels)
    ch = calinski_harabasz_score(space, labels)
    db = davies_bouldin_score(space, labels)
    ari = adjusted_rand_score(true_cultivars, labels)
    print(f'{name:30s} | Sil={sil:.4f} | CH={ch:6.2f} | DB={db:.4f} | ARI={ari:.4f}')

## 5. Optional Supervised Learning Benchmark: Global vs. Cluster-Specific Models
Testing whether applying separate Ridge Regression models to individual K-Means clusters improves prediction of `color_intensity` compared to a single global model.

In [5]:
km_labels = models['K-Means (16D, k=3)']
predictors = [c for c in raw.feature_names if c != 'color_intensity']
X_sup = StandardScaler().fit_transform(df_clean[predictors].values)
y_sup = df_clean['color_intensity'].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_global = cross_val_predict(Ridge(alpha=1.0), X_sup, y_sup, cv=kf)

y_local = np.zeros_like(y_sup)
for cid in range(3):
    m = km_labels == cid
    y_local[m] = cross_val_predict(Ridge(alpha=1.0), X_sup[m], y_sup[m], cv=KFold(n_splits=5, shuffle=True, random_state=42))

print(f'Global Model R2: {r2_score(y_sup, y_global):.4f}, RMSE: {np.sqrt(mean_squared_error(y_sup, y_global)):.4f}')
print(f'Cluster-Local R2: {r2_score(y_sup, y_local):.4f}, RMSE: {np.sqrt(mean_squared_error(y_sup, y_local)):.4f}')